In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
import subprocess
import time

# Step 1 - Install zstd first
subprocess.run("apt-get install -y zstd", shell=True, check=True)

# Step 2 - install ollama
subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    check=True
)

# Step 3 - Start ollama server in background
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(15)  # Wait for server to start

# Step 4 - Pull the model
subprocess.run(["ollama", "pull", "llama3.2:3b"], check=True)

print("Ollama is ready!")

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (12.0 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%#######                                           43.2%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 6.7 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   4% ▕                  ▏  83 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   9% ▕█                 ▏ 175 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  11% ▕█                 ▏ 223 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  16% ▕██                ▏ 313 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  20% ▕███               ▏ 409 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  23% ▕████              ▏ 456 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  27% ▕████              ▏ 549 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  32% ▕█████             ▏ 642 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  34% ▕█████

Ollama is ready!


pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest ⠦ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B    

In [4]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [5]:
import ollama

response = ollama.chat(
    model='llama3.2:3b',
    messages=[{'role': 'user', 'content': 'Say hello in one sentence'}]
)

print(response['message']['content'])

Hello!


In [6]:
# Load dataset (2WikiMultihopQA dev)
import requests, zipfile, io, json

url = "https://gitlab.tudelft.nl/anonymous_arr/bcqa_data/-/raw/main/2wikimultihopQA.zip"
response = requests.get(url)

z = zipfile.ZipFile(io.BytesIO(response.content))

with z.open("2wikimultihopQA/dev.json") as f:
    data = json.load(f)

# take 500 queries
data = data[:500]

print("Loaded queries:", len(data))

Loaded queries: 500


In [7]:
pip install gdown

Note: you may need to restart the kernel to use updated packages.


In [8]:
import gdown

url = "https://drive.google.com/uc?id=1aIyo583DCy3oAuQSgvIKf_SUnV3PEAer"
output = "bm25_2wiki.json"

gdown.download(url, output, quiet=False)

# load
with open(output) as f:
    bm25_data = json.load(f)

print(type(bm25_data))
print(len(bm25_data))

# check structure
first_key = list(bm25_data.keys())[0]
print("Example key:", first_key)
print("Example docs:", bm25_data[first_key][:2])

Downloading...
From: https://drive.google.com/uc?id=1aIyo583DCy3oAuQSgvIKf_SUnV3PEAer
To: /kaggle/working/bm25_2wiki.json
100%|██████████| 52.7M/52.7M [00:00<00:00, 171MB/s] 


<class 'dict'>
1200
Example key: 8813f87c0bdd11eba7f7acde48001122
Example docs: ['Polish-Russian War (Wojna polsko-ruska) is a 2009 Polish film directed by Xawery Żuławski based on the novel Polish-Russian War under the white-red flag by Dorota Masłowska.', 'Polish- Russian War( Wojna polsko- ruska) is a 2009 Polish film directed by Xawery Żuławski based on the novel Polish- Russian War under the white- red flag by Dorota Masłowska.']


In [9]:
# Check if dataset IDs match BM25 keys

sample_item = data[0]

print("Dataset ID:", sample_item.get("_id", sample_item.get("id")))
print("Exists in BM25:", sample_item.get("_id", sample_item.get("id")) in bm25_data)

# Check a few samples
for i in range(5):
    qid = data[i].get("_id", data[i].get("id"))
    print(qid, "->", qid in bm25_data)

Dataset ID: 8813f87c0bdd11eba7f7acde48001122
Exists in BM25: True
8813f87c0bdd11eba7f7acde48001122 -> True
61a46987092f11ebbdaeac1f6bf848b6 -> True
e2a3bf2a0bdd11eba7f7acde48001122 -> True
0cd3bdea0bde11eba7f7acde48001122 -> True
f9dcb4a60bda11eba7f7acde48001122 -> True


In [12]:
pip install -U transformers sentence-transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 87.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 88.3 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: tran

In [10]:
from sentence_transformers import CrossEncoder

cross_enc = CrossEncoder("nreimers/mmarco-mMiniLMv2-L12-H384-v1",device="cuda",trust_remote_code=True)

config.json:   0%|          | 0.00/813 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: nreimers/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [11]:
def rerank_docs(query, docs, top_k=5):
    pairs = [(query, doc) for doc in docs]
    scores = cross_enc.predict(pairs)
    ranked = sorted(zip(scores, docs), reverse=True)
    return [doc for _, doc in ranked[:top_k]]

print("CrossEncoder ready (GPU)")

CrossEncoder ready (GPU)


In [12]:
import re
# Baseline follow-up generation (plain prompting)

def ensure_question(text, query):
    text = str(text).strip()
    text = text.replace("Question:", "").replace("Follow-up", "").replace(":", "").strip()
    text = " ".join(text.split())

    if not text:
        return f"What specific information are you looking for about {query}?"

    if "?" in text:
        text = text.split("?")[0].strip() + "?"
        return text

    lowered = text.lower()
    if lowered.startswith(("what ", "which ", "who ", "where ", "when ", "why ", "how ", "are ", "is ", "do ", "does ", "did ", "can ", "could ", "would ", "should ")):
        return text.rstrip(".!") + "?"

    if lowered.endswith(" is") or lowered.endswith(" are"):
        return text.rstrip(".!") + "?"

    return f"What specific information are you looking for about {query}?"


def generate_baseline_followup(query):
    prompt = f"""
    Generate ONE useful follow-up question to clarify the query.

    Rules:
    - Ask about a specific missing detail (e.g., location, type, features, time, etc.)
    - Do NOT ask vague questions like "What do you want to know?"
    - The question must narrow down the query and be relevant to the original query and potential information needs.
    - Frame the output as a question.
    - The output must end with a question mark.
    - Do not answer the query.
    - Do not complete the sentence as a statement.

    Query: {query}

    Return ONLY the question.
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 50, 'temperature': 0}
    )

    msg = response['message']
    text = msg.get('content', '').strip()

    return ensure_question(text, query)


In [13]:
# Baseline: Decomposition (Parallel)
def generate_decomposition_followup(query):
    prompt = f"""
    Generate 3 different follow-up questions about the query, each focusing on a different aspect.

    Examples of aspects:
    - what/who it is
    - role, function, or purpose
    - history, location, or usage

    Rules:
    - Each question must be specific and useful
    - Do NOT ask generic questions like "What do you want to know?"
    - Do NOT repeat the same idea
    - Keep questions simple and relevant

    Query: {query}

    Return ONLY 3 questions, each on a new line.
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 100, 'temperature': 0}
    )

    text = response['message']['content'].strip()

    # split lines and pick first valid question
    lines = [re.sub(r"^\d+[\.\)]\s*", "", l.strip()) for l in text.split("\n") if l.strip()]

    valid = []
    for l in lines:
        q = ensure_question(l, query)

        # remove generic fallback
        if "what specific information" in q.lower():
            continue

        # avoid super short junk
        if len(q.split()) < 4:
            continue

        valid.append(q)

    if valid:
        return valid[0]

    return ensure_question("", query)

In [14]:
# Baseline: Yes/No (Claim-style)
def generate_yesno_followup(query):
    prompt = f"""
    Generate ONE yes/no follow-up question to clarify the query.

    Rules:
    - The question must be answerable with ONLY "yes" or "no"
    - Do NOT ask multiple-choice questions
    - Do NOT include "or" comparisons (e.g., person, place, or thing)
    - Do NOT assume specific facts
    - Keep it simple and relevant

    Query: {query}

    Return ONLY the question.
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 50, 'temperature': 0}
    )

    text = response['message']['content'].strip()

    if " or " in text.lower():
        return ensure_question("", query)

    return ensure_question(text, query)

In [15]:
def extract_aspects(query):
    import re

    query_clean = str(query).strip()
    query_lower = query_clean.lower()
    words = [w for w in re.findall(r"[a-zA-Z0-9]+", query_clean) if len(w) > 1]

    def has_any(phrases):
        return any(p in query_lower for p in phrases)

    def classify_query_type(text):
        text_lower = text.lower().strip()
        tokens = [w for w in re.findall(r"[a-zA-Z0-9]+", text_lower) if len(w) > 1]

        if len(tokens) <= 2:
            return "ambiguous_short_query"

        if has_any(["family tree", "ancestry", "genealogy"]):
            return "family_relationship"

        if has_any(["hotel", "resort", "inn", "airport", "terminal", "airlines"]):
            return "place_or_facility"

        if has_any(["restaurant", "museum", "university", "company", "organization", "school", "farm"]):
            return "organization_or_place"

        if has_any(["cheap", "best", "budget", "affordable", "price", "pricing", "cost", "provider", "plan"]):
            return "service_or_product"

        if has_any(["map", "located", "location", "address", "directions", "where is"]):
            return "place_or_location"

        if has_any(["who won", "biography", "born", "died", "who is", "who was"]):
            return "person"

        # Broad person-name heuristic without memorizing topic-specific exceptions.
        title_cased = [part for part in re.split(r"\s+", text.strip()) if part]
        if 2 <= len(title_cased) <= 4 and all(part[:1].isalpha() for part in title_cased):
            non_person_markers = {
                "airport", "hotel", "resort", "museum", "restaurant", "company",
                "university", "school", "farm", "lake", "park", "internet",
                "family", "tree", "map", "disease", "county", "tourism"
            }
            if not any(token in non_person_markers for token in tokens):
                return "person"

        if has_any(["disease", "symptoms", "treatment", "causes"]):
            return "topic_or_condition"

        if has_any(["album", "movie", "film", "book", "song", "game"]):
            return "work_or_media"

        return "generic"

    query_type = classify_query_type(query_clean)

    aspect_priors = {
        "family_relationship": ["family members", "ancestry", "relationships"],
        "service_or_product": ["type or option", "pricing or availability", "features or providers"],
        "place_or_facility": ["location", "services or amenities", "current status"],
        "place_or_location": ["location", "key features", "relevance or use"],
        "organization_or_place": ["main purpose or type", "location", "notable features"],
        "person": ["identity", "role or career", "notable contribution"],
        "topic_or_condition": ["definition or type", "causes or features", "diagnosis or treatment"],
        "work_or_media": ["work identity", "content or subject", "notable significance"],
        "ambiguous_short_query": ["intended meaning", "context", "specific need"],
    }

    if query_type in aspect_priors:
        return aspect_priors[query_type]

    prompt = f"""
    Identify 3 key aspects needed to answer this query well.

    An aspect is a dimension of information required to satisfy the query,
    including implicit missing information when relevant.

    Rules:
    - exactly 3 aspects
    - short phrases only
    - 2 to 5 words each
    - no full sentences
    - avoid generic items like "details", "information", "major achievements", or "historical significance"
    - prefer aspects that help generate useful follow-up questions for this specific query type
    - if the query is about a person, use person-style aspects only when appropriate
    - if the query is about a place, organization, service, or object, do NOT use person-style aspects

    Query: {query_clean}

    Aspects:
    """
    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 80, 'temperature': 0}
    )

    def clean_aspects(text):
        lines = text.split("\n")
        cleaned = []
        banned = {
            "details", "information", "general information", "overview", "background",
            "major achievements", "historical significance"
        }

        for line in lines:
            line = line.strip()
            line = line.lstrip("1234567890.-) ").strip()
            line = re.sub(r'[^a-zA-Z0-9\s/&-]', '', line).strip()

            if not line:
                continue
            if len(line.split()) < 1 or len(line.split()) > 6:
                continue
            if line.lower() in banned:
                continue
            if any(x in line.lower() for x in ["part", "definition", "usage"]):
                continue

            cleaned.append(line)

        deduped = []
        seen = set()
        for item in cleaned:
            key = item.lower()
            if key not in seen:
                seen.add(key)
                deduped.append(item)

        return deduped[:3]

    raw = response['message']['content']
    aspects = clean_aspects(raw)

    if len(aspects) < 3:
        fallback = ["main topic", "key facts", "context"]
        for item in fallback:
            if len(aspects) < 3 and item.lower() not in {a.lower() for a in aspects}:
                aspects.append(item)

    return aspects[:3]

In [16]:
# detect ambiguity BEFORE answer generation
def detect_ambiguity(query):
    query = query.strip()

    # well-formed named entities usually do not need intent clarification
    if len(query.split()) >= 3 and query.replace(" ", "").isalpha():
        return False

    prompt = f"""
    Query: {query}

    Does this query have multiple meanings or interpretations,
    or is it missing the intent needed to answer it correctly?

    Answer only: YES or NO
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 5, 'temperature': 0}
    )

    return "YES" in response['message']['content'].upper()

In [17]:
# Missing aspect identification
def find_missing_aspect(query, answer, aspects, allow_intent=False):

    def normalize(text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    answer_norm = normalize(answer)

    if allow_intent and detect_ambiguity(query):
        return "INTENT"

    weak_answer = (
        not answer_norm
        or "insufficient information" in answer_norm
        or "ambiguous query" in answer_norm
        or len(answer_norm.split()) < 12
        or not str(answer).strip().endswith((".", "?", "!"))
    )

    if not aspects:
        return "DETAIL" if weak_answer else "NONE"

    stopwords = {"and", "of", "the", "to", "from", "for", "in", "on", "with", "or"}
    coverage = {}
    uncovered = []

    for aspect in aspects:
        aspect_norm = normalize(aspect)
        aspect_words = [w for w in aspect_norm.split() if w not in stopwords and len(w) > 2]

        if not aspect_words:
            coverage[aspect] = 0.0
            uncovered.append(aspect)
            continue

        matched = sum(1 for w in aspect_words if w in answer_norm)
        ratio = matched / max(len(aspect_words), 1)
        coverage[aspect] = ratio

        if ratio < 0.6:
            uncovered.append(aspect)

    if not uncovered:
        return "DETAIL" if weak_answer else "NONE"

    if weak_answer and len(uncovered) == len(aspects):
        return uncovered[0]

    prompt = f"""
    Original query: {query}

    Combined answer so far: {answer}

    Candidate aspects and approximate coverage:
    {coverage}

    Remaining candidate aspects:
    {uncovered}

    Choose the ONE most important aspect still missing to answer the original query well.

    Rules:
    - Return NONE only if the answer is already sufficient.
    - Otherwise return exactly one aspect from the remaining candidate list.
    - Prefer the aspect whose absence most limits answering the original query.
    - Do not invent a new aspect.

    Output format:
    Missing: <NONE or exact aspect>
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 30, 'temperature': 0}
    )

    text = response['message']['content'].strip()
    if "Missing:" in text:
        missing = text.split("Missing:")[1].split("\n")[0].strip()
    else:
        missing = uncovered[0]

    if missing == "NONE":
        return "DETAIL" if weak_answer else "NONE"

    if missing in uncovered:
        return missing

    return uncovered[0]

In [18]:
# Improved follow-up generation
def generate_aspect_followup(query, missing_aspect, aspects=None, asked_followups=None):
    asked_followups = asked_followups or []
    asked_norm = {q.strip().lower() for q in asked_followups if q}

    if missing_aspect == "NONE":
        return None

    if missing_aspect == "INTENT":
        candidate = f"What specific information do you want to know about {query}?"
        return None if candidate.lower() in asked_norm else candidate

    if missing_aspect == "DETAIL":
        candidate = f"What specific detail about {query} are you looking for?"
        return None if candidate.lower() in asked_norm else candidate

    prompt = f"""
    Original query: {query}

    Missing aspect: {missing_aspect}

    Generate ONE clear, direct clarification question asking only about this missing aspect.

    Rules:
    - Start with What, Which, Who, Where, or When.
    - Keep it specific and factual.
    - Mention the original topic naturally.
    - Do not ask a generic question.
    - Do not repeat the aspect wording awkwardly.
    - Output only the question.
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 50, 'temperature': 0}
    )

    text = response['message']['content'].strip()
    if "?" in text:
        text = text.split("?")[0].strip() + "?"
    elif text:
        text = text.strip() + "?"

    if not text:
        text = f"What about {missing_aspect} is important for {query}?"

    if text.lower() in asked_norm:
        simple = f"What is {missing_aspect} for {query}?"
        if simple.lower() not in asked_norm:
            return simple
        return None

    return text

In [19]:
# generate answer
def generate_answer(question, docs, decomposition_path=None):
    import re

    reasoning_text = ""

    if decomposition_path:
        reasoning_text = "\n".join(decomposition_path)

    if not docs:
        return {
            "raw": "Insufficient information",
            "final": "Insufficient information"
        }

    # SAME evidence formatting style
    top_final = "\n".join(docs[:10])

    # EXACT system prompt style
    system_prompt_1 = (
        "Follow the given examples and Given the question and context, "
        "reasoning path, think step by step extract key segments from given evidence "
        "relevant to question and give rationale, by forming your own reasoning path "
        "preceded by [Answer]: and output final answer for the question using "
        "information from given evidences and give concise precise answer preceded "
        "by [Final Answer]:\n"
    )

    # SAME user prompt style
    user_prompt = """
    [Question]: When does monsoon season end in the state the area code 575 is located?
    [Answer]: The area code 575 is located in New Mexico. Monsoon season in New Mexico typically ends in mid-September.
    [Final Answer]: mid-September.

    [Question]: Who lived longer, Theodor Haecker or Harry Vaughan Watkins?
    [Answer]: Theodor Haecker was 65 years old when he died. Harry Vaughan Watkins was 69 years old when he died. Hence Harry Vaughan Watkins lived longer.
    [Final Answer]: Harry Vaughan Watkins.

    [Question]: What is the current official currency in the country where Ineabelle Diaz is a citizen?
    [Answer]: Ineabelle Diaz is from Puerto Rico, which is in the United States of America. The current official currency in the United States is the United States dollar.
    [Final Answer]: United States dollar.

    [Question]: Where was the person who founded the American Institute of Public Opinion in 1935 born?
    [Answer]: The person who founded the American Institute of Public Opinion in 1935 is George Gallup. George Gallup was born in Jefferson, Iowa.
    [Final Answer]: Jefferson.

    [Question]: What language is used by the director of Tiffany Memorandum?
    [Answer]: The director of Tiffany Memorandum is Sergio Grieco. Sergio Grieco speaks Italian.
    [Final Answer]: Italian.

    [Question]: What is the sports team the person played for who scored the first touchdown in Superbowl 1?
    [Answer]: The player that scored the first touchdown in Superbowl 1 is Max McGee. Max McGee played for the Green Bay Packers.
    [Final Answer]: Green Bay Packers.

    [Question]: The birth country of Jayantha Ketagoda left the British Empire when?
    [Answer]: The birth country of Jayantha Ketagoda is Sri Lanka. Sri Lanka left the British Empire on February 4, 1948.
    [Final Answer]: February 4, 1948.

    """ + (
            "Follow the above example and Given existing Reasoning path: "
            + reasoning_text
            + "\n the evidence, Evidence: "
            + top_final
            + "\n and use the most relevant information for the question "
            + "from the most relevant evidence from given Evidence: "
            + "and form your own correct reasoning path to derive the answer "
            + "thinking step by step preceded by [Answer]: "
            + "and subsequently give final answer as shown in above examples "
            + "preceded by [Final Answer]: for the Question: "
            + question
        )

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[
            {
                'role': 'system',
                'content': system_prompt_1
            },
            {
                'role': 'user',
                'content': user_prompt
            }
        ],
        options={
            'num_predict': 1000,
            'temperature': 0.2
        }
    )

    output = response['message'].get('content', '').strip()

    # safer extraction
    matches = re.findall(
        r"\[Final Answer\]:\s*(.+)",
        output,
        re.IGNORECASE
    )

    final_answer = matches[-1].strip() if matches else output.strip()

    return {
        "raw": output,
        "final": final_answer
    }

In [20]:
def run_iterative_pipeline_bm25(query, original_docs, max_steps=3):

    def normalize(text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    history = []
    all_docs = []
    base_aspects = extract_aspects(query)
    asked_followups = []
    resolved_aspects = []
    current_query = query

    for step in range(1, max_steps + 1):
        # Rerank FIRST using current query
        docs = rerank_docs(current_query, original_docs, top_k=5)

        # Collect docs for pooling
        all_docs.extend(docs)

        # Then generate answer
        history_text = []

        for h in history:
            if h.get("query"):
                history_text.append(f"Question: {h['query']}")
            if h.get("answer"):
                history_text.append(f"Intermediate Answer: {h['answer']}")

        answer_obj = generate_answer(
            current_query,
            docs,
            decomposition_path=history_text
        )

        answer = answer_obj["final"]
        raw_answer = answer_obj["raw"]

        combined_answer_parts = [
            item["answer"]
            for item in history
            if item.get("answer") and item["answer"] != "Insufficient information"
        ]
        if answer and answer != "Insufficient information":
            combined_answer_parts.append(answer)

        combined_answer = " ".join(combined_answer_parts).strip() or answer

        remaining_aspects = [a for a in base_aspects if a not in resolved_aspects]

        missing = find_missing_aspect(
            query,
            combined_answer,
            remaining_aspects,
            allow_intent=(step == 1)
        )

        step_record = {
            "step": step,
            "query": current_query,
            "answer": answer,
            "missing": missing,
            "followup": None
        }

        if missing not in ["NONE", "INTENT", "DETAIL"] and missing in remaining_aspects:
            resolved_aspects.append(missing)

        if missing == "NONE":
            history.append(step_record)
            break

        if current_query != query and answer == "Insufficient information":
            history.append(step_record)
            break

        if step >= max_steps:
            history.append(step_record)
            break

        followup = generate_aspect_followup(
            query,
            missing,
            remaining_aspects,
            asked_followups=asked_followups
        )

        if followup and "what specific information" in followup.lower():
            followup = f"What is the intended meaning of {query}?"

        if not followup:
            history.append(step_record)
            break

        if history:
            prev_followup = history[-1].get("followup")
            prev_answer = normalize(history[-1].get("answer", ""))
            curr_answer = normalize(answer)

            combined_prev = normalize(" ".join([
                item["answer"] for item in history
                if item.get("answer") and item["answer"] != "Insufficient information"
            ]))

            no_new_information = curr_answer and curr_answer in combined_prev
            repeated_followup = followup == prev_followup
            highly_similar_answer = (
                curr_answer[:140] == prev_answer[:140]
                if curr_answer and prev_answer else False
            )

            if repeated_followup or highly_similar_answer or no_new_information:
                history.append(step_record)
                break

        step_record["followup"] = followup
        history.append(step_record)

        asked_followups.append(followup)
        current_query = followup

    # Preserve order while removing duplicates
    seen = set()
    pooled_docs = []
    for d in all_docs:
        if d not in seen:
            seen.add(d)
            pooled_docs.append(d)

    # Re-rank pooled docs with original query
    reranked_pooled_docs = rerank_docs(query, pooled_docs)

    return history, docs, pooled_docs, reranked_pooled_docs

# Generate follow-up questions for qualitative evaluation

In [22]:
# filtered
import random
import pandas as pd

random.seed(42)

qual_results = []

# shuffle full dataset
shuffled_data = data.copy()
random.shuffle(shuffled_data)

for item in shuffled_data:

    # stop once we have 60 VALID samples
    if len(qual_results) >= 60:
        break

    query = item["question"]

    # -----------------------------
    # BASELINE FOLLOW-UP
    # -----------------------------
    try:
        baseline_q = generate_baseline_followup(query)
    except:
        baseline_q = ""

    # -----------------------------
    # ITERATIVE FOLLOW-UP
    # -----------------------------
    iterative_q = ""

    try:
        qid = item.get("_id", item.get("id"))
        raw_docs = bm25_data[qid]

        history, final_docs, pooled_docs, reranked_pooled_docs = run_iterative_pipeline_bm25(
            query,
            raw_docs
        )

        for h in reversed(history):

            f = h.get("followup")

            if not f:
                continue

            f_lower = f.lower()

            if "what is the intended meaning" in f_lower:
                continue

            if "what is the identity of" in f_lower:
                continue

            iterative_q = f
            break

    except:
        continue

    # skip rows without usable iterative followup
    if not iterative_q:
        continue

    # -----------------------------
    # STORE
    # -----------------------------
    qual_results.append({
        "original_query": query,
        "baseline_followup": baseline_q,
        "iterative_followup": iterative_q
    })

# dataframe
df_qual = pd.DataFrame(qual_results)

# export
df_qual.to_excel(
    "qualitative_followups.xlsx",
    index=False
)

print(df_qual.head())
print(f"\nCollected {len(df_qual)} valid samples")

                                      original_query  \
0          When was the director of film Jinpa born?   
1  Which film has the director born earlier, Idol...   
2  Where does the director of film Man At Bath wo...   
3  Are both Nikolsk Airport and Svobodny Airport ...   
4  Are both businesses, Telus and Ztr Control Sys...   

                                   baseline_followup  \
0          Was Jinpa's birthdate publicly disclosed?   
1  Is the birth year of the directors for both fi...   
2  Is the director of film "Man At Bath" a well-k...   
3  Are both airports part of the Russian Federal ...   
4                  Are they both Canadian companies?   

                                  iterative_followup  
0                          What is Jinpa's identity?  
1  What is the birth year of the directors of Ido...  
2  What is the director of film Man At Bath's cur...  
3  What amenities are available at both Nikolsk A...  
4  What are the countries of origin for Telus and..

In [23]:
# unfiltered
import random
import pandas as pd

random.seed(42)

qual_results = []

# shuffle full dataset
shuffled_data = data.copy()
random.shuffle(shuffled_data)

for item in shuffled_data:

    # stop once we have enough samples
    if len(qual_results) >= 200:
        break

    query = item["question"]

    # -----------------------------
    # BASELINE FOLLOW-UP
    # -----------------------------
    try:
        baseline_q = generate_baseline_followup(query)
    except:
        baseline_q = ""

    # -----------------------------
    # ITERATIVE FOLLOW-UP
    # -----------------------------
    iterative_q = ""

    try:
        qid = item.get("_id", item.get("id"))
        raw_docs = bm25_data[qid]

        history, final_docs, pooled_docs, reranked_pooled_docs = run_iterative_pipeline_bm25(
            query,
            raw_docs
        )

        # get LAST available follow-up
        for h in reversed(history):

            f = h.get("followup")

            if not f:
                continue

            iterative_q = f
            break

    except:
        continue

    # skip rows with no follow-up at all
    if not iterative_q:
        continue

    # -----------------------------
    # STORE
    # -----------------------------
    qual_results.append({
        "original_query": query,
        "baseline_followup": baseline_q,
        "iterative_followup": iterative_q
    })

# dataframe
df_qual = pd.DataFrame(qual_results)

# export
df_qual.to_excel(
    "qualitative_followups_unfiltered.xlsx",
    index=False
)

print(df_qual.head())
print(f"\nCollected {len(df_qual)} samples")

                                      original_query  \
0          When was the director of film Jinpa born?   
1  Which film has the director born earlier, Idol...   
2  Which film has the director who was born later...   
3  Which film has the director died first, Le Gui...   
4  Where does the director of film Man At Bath wo...   

                                   baseline_followup  \
0          Was Jinpa's birthdate publicly disclosed?   
1  Is the birth year of the directors for both fi...   
2  Is the director's birth year specified in eith...   
3  Is there any information available on the dire...   
4  Is the director of film "Man At Bath" a well-k...   

                                  iterative_followup  
0                          What is Jinpa's identity?  
1  What is the birth year of the directors of Ido...  
2  What is the intended meaning of Which film has...  
3  What is the identity of the director for each ...  
4  What is the intended meaning of Where does the..

# Examples and Survey Set

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# -----------------------------------
# LOAD EXCEL
# -----------------------------------
df = pd.read_excel("qualitative_followups_unfiltered.xlsx")

# -----------------------------------
# USE THE FOLLOW-UP COLUMN
# -----------------------------------
QUESTION_COLUMN = "iterative_followup"

# -----------------------------------
# CREATE LABELS AUTOMATICALLY
# -----------------------------------
df["question_type"] = df[QUESTION_COLUMN].str.lower().apply(
    lambda x: "intended_meaning"
    if "what is the intended meaning of" in str(x)
    else "regular"
)

# -----------------------------------
# CREATE 20 EXAMPLES
# Stratified sampling preserves distribution
# -----------------------------------
remaining_df, examples_df = train_test_split(
    df,
    test_size=20,
    stratify=df["question_type"],
    random_state=42
)

# -----------------------------------
# CREATE 50 SURVEY QUESTIONS
# -----------------------------------
_, survey_df = train_test_split(
    remaining_df,
    test_size=50,
    stratify=remaining_df["question_type"],
    random_state=42
)

# -----------------------------------
# EXPORT FILES
# -----------------------------------
examples_df.to_excel("examples_20.xlsx", index=False)
survey_df.to_excel("survey_50.xlsx", index=False)

# -----------------------------------
# OPTIONAL CHECK
# -----------------------------------
print("\nOriginal Distribution")
print(df["question_type"].value_counts(normalize=True))

print("\nExamples Distribution")
print(examples_df["question_type"].value_counts(normalize=True))

print("\nSurvey Distribution")
print(survey_df["question_type"].value_counts(normalize=True))

print("\nDone!")


Original Distribution
question_type
regular             0.605
intended_meaning    0.395
Name: proportion, dtype: float64

Examples Distribution
question_type
regular             0.6
intended_meaning    0.4
Name: proportion, dtype: float64

Survey Distribution
question_type
regular             0.6
intended_meaning    0.4
Name: proportion, dtype: float64

Done!


# Final survey set

In [5]:
import pandas as pd
import json

# Read Excel
df = pd.read_excel("survey_50.xlsx")

sections = []

for _, row in df.iterrows():

    query = str(row["original_query"]).strip()
    baseline = str(row["baseline_followup"]).strip()
    iterative = str(row["iterative_followup"]).strip()

    sections.append({
        "query": query,
        "questions": [
            {
                "method": "Baseline",
                "text": baseline
            },
            {
                "method": "Iterative",
                "text": iterative
            }
        ]
    })

# Convert to JS format
js_output = "const SECTIONS = " + json.dumps(
    sections,
    ensure_ascii=False,
    indent=2
) + ";"

# Save
with open("sections.js", "w", encoding="utf-8") as f:
    f.write(js_output)

print("Done. File saved as sections.js")

Done. File saved as sections.js


# Fleiss Kappa

In [1]:
import pandas as pd
from statsmodels.stats.inter_rater import fleiss_kappa

# Load CSV
df = pd.read_csv("Qualitative Survey New - Sheet1.csv")

# Create unique item id
df["item_id"] = (
    df["Query"].astype(str) + " || " +
    df["Method"].astype(str) + " || " +
    df["Question"].astype(str)
)

def build_matrix(df, score_column):
    matrix = []

    grouped = df.groupby("item_id")

    for item, group in grouped:
        counts = [0, 0, 0, 0, 0]  # ratings 1-5

        for score in group[score_column]:
            score = int(score)
            counts[score - 1] += 1

        matrix.append(counts)

    return matrix

# Compute kappas
fluency_matrix = build_matrix(df, "Fluency")
relevance_matrix = build_matrix(df, "Relevance")
usefulness_matrix = build_matrix(df, "Usefulness")

fluency_kappa = fleiss_kappa(fluency_matrix)
relevance_kappa = fleiss_kappa(relevance_matrix)
usefulness_kappa = fleiss_kappa(usefulness_matrix)

print("Fluency Fleiss Kappa:", fluency_kappa)
print("Relevance Fleiss Kappa:", relevance_kappa)
print("Usefulness Fleiss Kappa:", usefulness_kappa)

Fluency Fleiss Kappa: 0.7846084810410592
Relevance Fleiss Kappa: 0.4742782464060829
Usefulness Fleiss Kappa: 0.4622659002410533
